# 03 - Model Serving with KServe

Deploys and tests InferenceServices using pre-defined ServingRuntimes:

| Service | ServingRuntime | Model | Protocol |
|---------|---------------|-------|---------|
| `smartshop-rec` | `smartshop-rec-runtime` | Two-Tower recommendation | REST |
| `smartshop-llm` | `smartshop-vllm-runtime` | Mistral-7B + LoRA via vLLM | OpenAI-compat |
| `smartshop-rag` | `smartshop-rec-runtime` | RAG Q&A (Feast + LLM) | REST |

**Prerequisites:**
- Trained models in S3 (run notebooks 01 & 02)
- KServe operator installed
- ServingRuntimes applied: `oc apply -f infrastructure/openshift/serving-runtimes.yaml`

In [ ]:
%pip install -q kserve kubernetes requests

In [ ]:
# Papermill parameters
NAMESPACE = "smartshop"
MINIO_ENDPOINT = "http://minio.smartshop.svc.cluster.local:9000"
MODEL_OUTPUT_DIR = "s3://smartshop-models/recommendation"
LLM_ADAPTER_DIR = "s3://smartshop-models/llm-adapter"
LLM_BASE_MODEL = "mistralai/Mistral-7B-Instruct-v0.3"
VLLM_RUNTIME = "smartshop-vllm-runtime"
REC_RUNTIME = "smartshop-rec-runtime"
REC_SERVER_IMAGE = "quay.io/abdhumal/smartshop-rec-server:latest"
FEAST_REPO_PATH = "/feast/feature_repo"
REDIS_HOST = "redis.smartshop.svc.cluster.local"
REDIS_PORT = "6379"
MILVUS_HOST = "milvus.smartshop.svc.cluster.local"
MILVUS_PORT = "19530"
CLUSTER_DOMAIN = ""
TIMEOUT_SECONDS = 900
SKIP_LLM = False
SKIP_RAG = False

## Authentication

In [ ]:
import os, subprocess

try:
    K8S_TOKEN = subprocess.check_output(['oc', 'whoami', '--show-token'], text=True).strip()
except Exception:
    K8S_TOKEN = os.getenv('K8S_TOKEN', '')
    if not K8S_TOKEN:
        token_path = '/var/run/secrets/kubernetes.io/serviceaccount/token'
        if os.path.exists(token_path):
            with open(token_path) as f:
                K8S_TOKEN = f.read().strip()

# Load kube config
from kubernetes import client as k8s_client, config
try:
    config.load_incluster_config()
    print('Loaded in-cluster config')
except:
    config.load_kube_config()
    print('Loaded local kube config')

## Deploy InferenceServices via KServe SDK

In [ ]:
from kserve import KServeClient, V1beta1InferenceService, V1beta1InferenceServiceSpec, V1beta1PredictorSpec
from kubernetes.client import (
    V1Container, V1ResourceRequirements, V1VolumeMount, V1Volume,
    V1EnvVar, V1EnvVarSource, V1SecretKeySelector,
    V1ConfigMapVolumeSource, V1ObjectMeta, V1ContainerPort
)

kserve_client = KServeClient()
print('KServeClient initialized')

### Recommendation InferenceService

In [ ]:
def secret_env(name, secret_name, key):
    return V1EnvVar(name=name, value_from=V1EnvVarSource(
        secret_key_ref=V1SecretKeySelector(name=secret_name, key=key)))

rec_isvc = V1beta1InferenceService(
    api_version='serving.kserve.io/v1beta1',
    kind='InferenceService',
    metadata=V1ObjectMeta(
        name='smartshop-rec',
        namespace=NAMESPACE,
        labels={'app': 'smartshop'},
        annotations={
            'serving.kserve.io/autoscalerClass': 'hpa',
            'serving.kserve.io/targetUtilizationPercentage': '80',
        }
    ),
    spec=V1beta1InferenceServiceSpec(
        predictor=V1beta1PredictorSpec(
            min_replicas=1,
            max_replicas=3,
            containers=[
                V1Container(
                    name='kserve-container',
                    image=REC_SERVER_IMAGE,
                    ports=[V1ContainerPort(container_port=8000, protocol='TCP')],
                    env=[
                        V1EnvVar(name='MODEL_PATH', value=MODEL_OUTPUT_DIR),
                        V1EnvVar(name='FEAST_REPO_PATH', value=FEAST_REPO_PATH),
                        V1EnvVar(name='REDIS_HOST', value=REDIS_HOST),
                        V1EnvVar(name='REDIS_PORT', value=REDIS_PORT),
                        V1EnvVar(name='AWS_ENDPOINT_URL_S3', value=MINIO_ENDPOINT),
                        V1EnvVar(name='AWS_DEFAULT_REGION', value='us-east-1'),
                        secret_env('AWS_ACCESS_KEY_ID', 'smartshop-credentials', 'AWS_ACCESS_KEY_ID'),
                        secret_env('AWS_SECRET_ACCESS_KEY', 'smartshop-credentials', 'AWS_SECRET_ACCESS_KEY'),
                    ],
                    resources=V1ResourceRequirements(
                        requests={'cpu': '2', 'memory': '4Gi'},
                        limits={'cpu': '4', 'memory': '8Gi'}
                    ),
                )
            ]
        )
    )
)

try:
    kserve_client.get('smartshop-rec', namespace=NAMESPACE)
    print('smartshop-rec exists, replacing...')
    kserve_client.replace('smartshop-rec', rec_isvc, namespace=NAMESPACE)
except Exception:
    print('Creating smartshop-rec...')
    kserve_client.create(rec_isvc)

print('smartshop-rec submitted')

### LLM InferenceService (vLLM via ServingRuntime)

In [ ]:
if SKIP_LLM:
    print('LLM deployment skipped (SKIP_LLM=True)')
else:
    llm_isvc = V1beta1InferenceService(
        api_version='serving.kserve.io/v1beta1',
        kind='InferenceService',
        metadata=V1ObjectMeta(
            name='smartshop-llm',
            namespace=NAMESPACE,
            labels={'app': 'smartshop'},
            annotations={
                'serving.kserve.io/autoscalerClass': 'hpa',
            }
        ),
        spec=V1beta1InferenceServiceSpec(
            predictor=V1beta1PredictorSpec(
                min_replicas=1,
                max_replicas=2,
                model={
                    'modelFormat': {'name': 'vLLM'},
                    'runtime': VLLM_RUNTIME,
                    'storageUri': f'hf://{LLM_BASE_MODEL}',
                    'args': [
                        '--enable-lora',
                        '--lora-modules', f'smartshop-adapter={LLM_ADAPTER_DIR}',
                    ],
                    'env': [
                        {'name': 'HF_TOKEN', 'valueFrom': {'secretKeyRef': {'name': 'hf-credentials', 'key': 'token'}}},
                        {'name': 'AWS_ENDPOINT_URL_S3', 'value': MINIO_ENDPOINT},
                        {'name': 'AWS_DEFAULT_REGION', 'value': 'us-east-1'},
                        {'name': 'AWS_ACCESS_KEY_ID', 'valueFrom': {'secretKeyRef': {'name': 'smartshop-credentials', 'key': 'AWS_ACCESS_KEY_ID'}}},
                        {'name': 'AWS_SECRET_ACCESS_KEY', 'valueFrom': {'secretKeyRef': {'name': 'smartshop-credentials', 'key': 'AWS_SECRET_ACCESS_KEY'}}},
                    ],
                },
            )
        )
    )

    try:
        kserve_client.get('smartshop-llm', namespace=NAMESPACE)
        print('smartshop-llm exists, replacing...')
        kserve_client.replace('smartshop-llm', llm_isvc, namespace=NAMESPACE)
    except Exception:
        print('Creating smartshop-llm...')
        kserve_client.create(llm_isvc)

    print('smartshop-llm submitted')

### RAG InferenceService

In [ ]:
if SKIP_RAG:
    print('RAG deployment skipped (SKIP_RAG=True)')
else:
    rag_isvc = V1beta1InferenceService(
        api_version='serving.kserve.io/v1beta1',
        kind='InferenceService',
        metadata=V1ObjectMeta(
            name='smartshop-rag',
            namespace=NAMESPACE,
            labels={'app': 'smartshop'},
        ),
        spec=V1beta1InferenceServiceSpec(
            predictor=V1beta1PredictorSpec(
                min_replicas=1,
                max_replicas=3,
                containers=[
                    V1Container(
                        name='kserve-container',
                        image=REC_SERVER_IMAGE,
                        ports=[V1ContainerPort(container_port=8002, protocol='TCP')],
                        env=[
                            V1EnvVar(name='SERVE_MODE', value='rag'),
                            V1EnvVar(name='FEAST_REPO_PATH', value=FEAST_REPO_PATH),
                            V1EnvVar(name='LLM_URL', value=f'http://smartshop-llm.{NAMESPACE}.svc.cluster.local/v1/completions'),
                            V1EnvVar(name='REDIS_HOST', value=REDIS_HOST),
                            V1EnvVar(name='REDIS_PORT', value=REDIS_PORT),
                            V1EnvVar(name='MILVUS_HOST', value=MILVUS_HOST),
                            V1EnvVar(name='MILVUS_PORT', value=MILVUS_PORT),
                            V1EnvVar(name='AWS_ENDPOINT_URL_S3', value=MINIO_ENDPOINT),
                            V1EnvVar(name='AWS_DEFAULT_REGION', value='us-east-1'),
                            secret_env('AWS_ACCESS_KEY_ID', 'smartshop-credentials', 'AWS_ACCESS_KEY_ID'),
                            secret_env('AWS_SECRET_ACCESS_KEY', 'smartshop-credentials', 'AWS_SECRET_ACCESS_KEY'),
                        ],
                        resources=V1ResourceRequirements(
                            requests={'cpu': '2', 'memory': '4Gi'},
                            limits={'cpu': '4', 'memory': '8Gi'}
                        ),
                    )
                ]
            )
        )
    )

    try:
        kserve_client.get('smartshop-rag', namespace=NAMESPACE)
        print('smartshop-rag exists, replacing...')
        kserve_client.replace('smartshop-rag', rag_isvc, namespace=NAMESPACE)
    except Exception:
        print('Creating smartshop-rag...')
        kserve_client.create(rag_isvc)

    print('smartshop-rag submitted')

## Wait for InferenceServices to become Ready

In [ ]:
import time

services = ['smartshop-rec']
if not SKIP_LLM:
    services.append('smartshop-llm')
if not SKIP_RAG:
    services.append('smartshop-rag')

for svc in services:
    print(f'Waiting for {svc}...')
    try:
        kserve_client.wait_isvc_ready(svc, namespace=NAMESPACE, timeout_seconds=TIMEOUT_SECONDS)
        print(f'  {svc}: READY')
    except Exception as e:
        print(f'  {svc}: TIMEOUT/ERROR — {e}')
        raise

print('\nAll InferenceServices ready!')

## Smoke-test: Recommendation endpoint

In [ ]:
import requests, json, time

REC_ENDPOINT = f"http://smartshop-rec.{NAMESPACE}.svc.cluster.local"

payload = {'user_id': 42, 'n': 5}
try:
    t0 = time.time()
    resp = requests.post(f'{REC_ENDPOINT}/recommend', json=payload, timeout=30)
    latency = (time.time() - t0) * 1000
    print(f'Status: {resp.status_code} | Latency: {latency:.0f}ms')
    print(f'Response: {json.dumps(resp.json(), indent=2)[:500]}')
    assert resp.status_code == 200
    print('\nRecommendation smoke test PASSED')
except requests.ConnectionError:
    print('Cannot reach internal URL (expected outside cluster)')
    print('Rec endpoint exists — manual test from within cluster required.')

## Smoke-test: LLM (vLLM) endpoint

In [ ]:
if SKIP_LLM:
    print('LLM test skipped')
else:
    LLM_ENDPOINT = f"http://smartshop-llm.{NAMESPACE}.svc.cluster.local"

    payload = {
        'model': 'smartshop-adapter',
        'prompt': '[INST] Summarize reviews for product: wireless headphones [/INST]',
        'max_tokens': 128,
        'temperature': 0.7
    }

    try:
        t0 = time.time()
        resp = requests.post(f'{LLM_ENDPOINT}/v1/completions', json=payload, timeout=60)
        latency = (time.time() - t0) * 1000
        print(f'Status: {resp.status_code} | Latency: {latency:.0f}ms')
        print(f'Response: {json.dumps(resp.json(), indent=2)[:500]}')
        assert resp.status_code == 200
        print('\nLLM smoke test PASSED')
    except requests.ConnectionError:
        print('Cannot reach LLM internal URL — manual test required.')

## Smoke-test: RAG endpoint

In [ ]:
if SKIP_RAG:
    print('RAG test skipped')
else:
    RAG_ENDPOINT = f"http://smartshop-rag.{NAMESPACE}.svc.cluster.local"

    payload = {
        'question': 'What do customers say about battery life of wireless earbuds?',
        'top_k': 3
    }

    try:
        t0 = time.time()
        resp = requests.post(f'{RAG_ENDPOINT}/ask', json=payload, timeout=60)
        latency = (time.time() - t0) * 1000
        print(f'Status: {resp.status_code} | Latency: {latency:.0f}ms')
        print(f'Response: {json.dumps(resp.json(), indent=2)[:500]}')
        assert resp.status_code == 200
        print('\nRAG smoke test PASSED')
    except requests.ConnectionError:
        print('Cannot reach RAG internal URL — manual test required.')

In [ ]:
print('NOTEBOOK_STATUS: SUCCESS')